# MiniCells Core Validation 001 — Knowledge Subsumption and Computational Reorganization

Formal run for the frozen `core-validation-001` protocol. The experiment uses sequential modular addition with no replay, a balanced-random-label control, Fourier restricted/excluded interventions, and no growth mechanisms.


In [ ]:
from pathlib import Path
import os, subprocess, sys, json

BRANCH = 'codex/core-validation-001-knowledge-subsumption'
REPO = 'https://github.com/ArcheLabs/mini-cells.git'
ROOT = Path('/kaggle/working/mini-cells')
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO, str(ROOT)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=ROOT, check=True)
os.chdir(ROOT)
print('HEAD', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('TREE', subprocess.check_output(['git', 'rev-parse', 'HEAD^{tree}'], text=True).strip())


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/research/04-continual-learning-core/test_core_validation_001.py'], check=True)
subprocess.run([sys.executable, 'scripts/research/run_core_validation_001.py', '--smoke', '--device', 'cpu', '--skip-oracle'], check=True)
print('Core Validation 001 tests and CPU smoke passed.')


In [ ]:
import torch
print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu_count': torch.cuda.device_count(), 'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})
assert torch.cuda.is_available(), 'Formal Core Validation 001 requires a CUDA GPU.'


In [ ]:
OUT = ROOT / 'results' / 'core-validation-001-knowledge-subsumption'
if OUT.exists():
    import shutil
    shutil.rmtree(OUT)
subprocess.run([sys.executable, 'scripts/research/run_core_validation_001.py', '--device', 'cuda'], check=True)
subprocess.run([sys.executable, 'scripts/research/report_core_validation_001.py'], check=True)


In [ ]:
decision = json.loads((OUT / 'decision.json').read_text())
print(json.dumps(decision, indent=2, sort_keys=True))
import pandas as pd
display(pd.read_csv(OUT / 'runs.csv'))


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(OUT / 'fourier-circuit-concentration.png')))
display(Image(filename=str(OUT / 'fourier-circuit-interventions.png')))
display(Image(filename=str(OUT / 'causal-path-reuse.png')))


## Publish

The final cell publishes curated formal results to `kaggle/core-validation-001-knowledge-subsumption-results`. It expects the existing Kaggle secret `GITHUB_TOKEN` with Contents read/write permission.


In [ ]:
subprocess.run([sys.executable, 'scripts/research/publish_core_validation_001.py', '--push'], check=True)
print('Published Core Validation 001 results branch.')
